# 6CS012 Worksheet 5 — End-to-End CNN for Amazon Fruit Classification
*(Alternative Implementation)*

## Setup

In [ ]:
import os, random, warnings, pathlib
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from PIL import Image
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers, callbacks
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, confusion_matrix

print("TensorFlow version:", tf.__version__)

BASE_DIR  = pathlib.Path("FruitinAmazon")
TRAIN_DIR = BASE_DIR / "train"
TEST_DIR  = BASE_DIR / "test"

IMG_SIZE    = (128, 128)
BATCH_SIZE  = 16
EPOCHS      = 250
VAL_SPLIT   = 0.2
SEED        = 42

## Task 1 — Data Understanding & Visualization
### 1a. Visualise one image per class

In [ ]:
# Use pathlib to discover classes
class_dirs = sorted([d for d in TRAIN_DIR.iterdir() if d.is_dir()])
classes = [d.name for d in class_dirs]
print(f"Classes ({len(classes)}): {classes}")
for d in class_dirs:
    print(f"  {d.name:12s}: {len(list(d.glob('*')))} images")

# Pick one random image per class and display using mpimg
n = len(classes)
cols = 3
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
fig.patch.set_facecolor('#0d0d0d')
axes = np.array(axes).flatten()

for i, (cls, d) in enumerate(zip(classes, class_dirs)):
    img_path = random.choice(list(d.glob("*")))
    img = mpimg.imread(str(img_path))
    axes[i].imshow(img)
    axes[i].set_title(cls, fontsize=13, fontweight='bold',
                      color='white', backgroundcolor='#c0392b', pad=6)
    axes[i].axis('off')

for ax in axes[n:]:
    ax.set_visible(False)

plt.suptitle("One Sample Per Fruit Class", fontsize=16, color='white', fontweight='bold')
plt.tight_layout()
plt.show()

**Observation:** Five Amazon fruit classes are present — acai, cupuacu, graviola, pupunha, and tucuma.
The dataset is very small (2–12 images per class) and imbalanced. Images differ in background colour,
scale, and photographic style, which makes generalisation harder for the CNN.

### 1b. Check for Corrupted Images

In [ ]:
# Use os.walk and Image.open().load() to fully decode each image
corrupted = []
for root, _, files in os.walk(TRAIN_DIR):
    for fname in files:
        fpath = os.path.join(root, fname)
        try:
            with Image.open(fpath) as img:
                img.load()   # force full decode
        except Exception:
            corrupted.append(fpath)
            os.remove(fpath)
            print(f"Removed corrupted image: {fpath}")

print("No Corrupted Images Found." if not corrupted
      else f"Total removed: {len(corrupted)} image(s).")

## Task 2 — Loading & Preprocessing Data

In [ ]:
# Use tf.keras.utils (newer API) instead of tf.keras.preprocessing
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    labels='inferred',
    label_mode='int',
    image_size=IMG_SIZE,
    interpolation='bilinear',       # different interpolation
    batch_size=BATCH_SIZE,
    shuffle=True,
    validation_split=VAL_SPLIT,
    subset='training',
    seed=SEED
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    labels='inferred',
    label_mode='int',
    image_size=IMG_SIZE,
    interpolation='bilinear',
    batch_size=BATCH_SIZE,
    shuffle=False,
    validation_split=VAL_SPLIT,
    subset='validation',
    seed=SEED
)

CLASS_NAMES = train_ds.class_names
NUM_CLASSES = len(CLASS_NAMES)
print(f"Class names : {CLASS_NAMES}")
print(f"Num classes : {NUM_CLASSES}")

# Normalise using lambda inside map (instead of Rescaling layer)
normalize = lambda x, y: (tf.cast(x, tf.float32) / 255.0, y)
AUTOTUNE  = tf.data.AUTOTUNE

train_ds = train_ds.map(normalize).cache().prefetch(AUTOTUNE)
val_ds   = val_ds.map(normalize).cache().prefetch(AUTOTUNE)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    labels='inferred',
    label_mode='int',
    image_size=IMG_SIZE,
    interpolation='bilinear',
    batch_size=BATCH_SIZE,
    shuffle=False,
    seed=SEED
)
test_class_names = test_ds.class_names
test_ds = test_ds.map(normalize).prefetch(AUTOTUNE)

## Task 3 — Build the CNN
*(Using the Functional API instead of Sequential)*

In [ ]:
# Build the same architecture via Functional API
inputs = keras.Input(shape=(*IMG_SIZE, 3), name="input_images")

# Data augmentation (only active during training)
x = layers.RandomFlip("horizontal")(inputs)
x = layers.RandomRotation(0.1)(x)

# Convolutional Block 1  (F=3×3, k=32, padding=same, stride=1)
x = layers.Conv2D(32, (3, 3), padding='same', activation='relu', name='conv1')(x)
x = layers.MaxPooling2D((2, 2), strides=2, name='pool1')(x)

# Convolutional Block 2  (F=3×3, k=32, padding=same, stride=1)
x = layers.Conv2D(32, (3, 3), padding='same', activation='relu', name='conv2')(x)
x = layers.MaxPooling2D((2, 2), strides=2, name='pool2')(x)

# Fully Connected Network
x = layers.Flatten(name='flatten')(x)
x = layers.Dense(64,  activation='relu', name='fc1')(x)
x = layers.Dense(128, activation='relu', name='fc2')(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax', name='output')(x)

model = keras.Model(inputs, outputs, name="FruitCNN_Functional")
model.summary()

## Task 4 — Compile & Train

In [ ]:
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cb_list = [
    callbacks.ModelCheckpoint(
        filepath='best_model_v2.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    callbacks.EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=7,
        min_lr=1e-6,
        verbose=1
    )
]

history = model.fit(
    train_ds,
    epochs=EPOCHS,
    validation_data=val_ds,
    callbacks=cb_list,
    verbose=1
)

## Task 5 — Evaluate the Model

In [ ]:
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_acc:.4f}  ({test_acc * 100:.1f}%)")

# Plot using history.history as a dict directly
hist = history.history
ep   = range(1, len(hist['accuracy']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#1a1a2e')

for ax in (ax1, ax2):
    ax.set_facecolor('#16213e')
    ax.tick_params(colors='white')
    ax.xaxis.label.set_color('white')
    ax.yaxis.label.set_color('white')
    ax.title.set_color('white')
    for s in ax.spines.values(): s.set_edgecolor('#e94560')

ax1.plot(ep, hist['accuracy'],     '#e94560', lw=2, label='Train Accuracy')
ax1.plot(ep, hist['val_accuracy'], '#00b4d8', lw=2, ls='--', label='Val Accuracy')
ax1.fill_between(ep, hist['accuracy'], hist['val_accuracy'], alpha=0.1, color='#e94560')
ax1.set_title('Accuracy over Epochs', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
ax1.legend(facecolor='#0f3460', labelcolor='white')

ax2.plot(ep, hist['loss'],     '#e94560', lw=2, label='Train Loss')
ax2.plot(ep, hist['val_loss'], '#00b4d8', lw=2, ls='--', label='Val Loss')
ax2.fill_between(ep, hist['loss'], hist['val_loss'], alpha=0.1, color='#00b4d8')
ax2.set_title('Loss over Epochs', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
ax2.legend(facecolor='#0f3460', labelcolor='white')

plt.suptitle('Training & Validation Curves', fontsize=15, fontweight='bold', color='white')
plt.tight_layout()
plt.show()

## Task 6 — Save & Load the Model

In [ ]:
# Save in SavedModel format (TF default) and also as .h5
model.save("fruit_cnn_model_v2.keras")
print("Saved as fruit_cnn_model_v2.keras")

model.save("fruit_cnn_model_v2.h5")
print("Saved as fruit_cnn_model_v2.h5")

# Reload and verify
loaded_model = keras.models.load_model("fruit_cnn_model_v2.h5")
r_loss, r_acc = loaded_model.evaluate(test_ds, verbose=0)
print(f"Reloaded model — Test Accuracy: {r_acc:.4f}")

## Task 7 — Predictions & Classification Report

In [ ]:
y_true, y_pred = [], []
for imgs, labels in test_ds:
    probs = loaded_model.predict(imgs, verbose=0)
    y_pred.extend(np.argmax(probs, axis=1))
    y_true.extend(labels.numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Classification report
print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=test_class_names, zero_division=0))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(7, 6))
fig.patch.set_facecolor('#1a1a2e')
ax.set_facecolor('#16213e')
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=test_class_names)
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix', fontsize=14, fontweight='bold', color='white')
ax.tick_params(colors='white')
ax.xaxis.label.set_color('white')
ax.yaxis.label.set_color('white')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()